In [1]:
# 랭체인에서 RAG(검색 + 생성) 구현, 분기체인(규칙기반 멀티 체인 선택)
!pip install langchain langchain-google-genai python-dotenv langchain-chroma langchain-community
!pip install sentence-transformers python-dotenv


In [18]:
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceBgeEmbeddings
import os
from dotenv import load_dotenv

load_dotenv()

# LLM 모델 준비
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.7)

# 임베딩 모델 준비
# embedding_model = GoogleGenerativeAIEmbeddings(model="models/embedding-001")    # 유료 서비스
embedding_model = HuggingFaceBgeEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 문서 데이터 준비
# 랭체인 RAG 구조는 문서들을 저장 -> 검색 -> 답변 생성 하는 흐름을 갖음
# 그런데 파일(.txt, .pdf ...)을 읽으면 구조가 복잡해 일단 메모리에 문서데이터를 만든다 (Document type)
# retriever는 텍스트 단위로 함
docs = [    # 주제가 다를 경우 Document를 따로따로 만들어준다(검색 효율 높이기 위함)
    Document(page_content="뉴욕증시 3대 지수가 동반 약세로 마감했습니다.미국 동부시간 20일 뉴욕증권거래소에서 다우지수는 전장보다 0.84%, S&P500 지수는 1.56%, 나스닥지수는 2.15% 각각 하락하며 마감했는데요.앞서 증시는 엔비디아 호실적에 힘입어 장 초반 급등했었습니다.\
    미국 9월 고용보고서가 연준의 금리 인하 기대감을 높인 것도 증시에 활력을 불어넣었는데요.미 노동부에 따르면 9월 실업률은 전달보다 0.1%포인트 오른 4.4%로 나타났습니다.다만 증시는 AI 거품 우려가 지속되며 기술주 투매 속에 약세로 반전했는데요.\
    연방준비제도 관계자가 금융자산에 대해 급락 위험 경고를 한 것도 증시에 약세 압력을 준 것으로 풀이됩니다.한편, 미국에서 2주 이상 일자리를 찾지 못한 실업수당 청구자가 팬데믹 이후 4년 만에 최대 수준으로 늘어난 것으로 나타났습니다.\
    미국 노동부는 2주 이상 실업수당을 신청한 '계속 실업수당' 청구 건수가 11월 2∼8일 주간 197만 4천 건으로 한 주 전보다 2만 8천 명 증가했다고 밝혔는데요.이번 통계는 연방정부 셧다운, 일시적 업무정지 사태가 끝난 뒤 처음 나온 것으로 실업 후 새 일자리를 바로 찾지 못하는 사람들이 늘었음을 의미합니다."),

    Document(page_content="‘99분 → 35분.’수도권광역급행철도(GTX) A노선 개통 후 서울 강남구 수서역에서 경기 화성 동탄역까지 가는 데 걸리는 시간이 1시간 넘게 단축됐다. GTX-A노선 정차역이 아닌 곳에서 출발하는 경우에도 통행시간이 14~35%가량 줄었다. 우여곡절 끝에 탄생한 새 대중교통이 톡톡한 효과를 내고 있다.\
    노선을 따라 부동산 시장도 들썩이고 있다. GTX 정차역과 가까울수록 집값이 더 많이 오르는 경향을 보였다. 역에서 500m 떨어진 거리에 있는 아파트가 2㎞ 떨어진 단지보다 50%포인트 더 높은 상승률을 기록했다. 개통 이후 일대 거래량도 증가하는 추세다.\
    최근 국토연구원이 발간한 ‘수도권 GTX-A노선(수서~동탄) 개통에 따른 영향 분석’ 자료에 따르면 수도권 동남권 일대 통행시간이 개통 전과 비교해 평균 20~65%가량 단축됐다. 일평균 통행량은 8488건으로 집계됐다. 버스, 지하철 등 대중교통을 이용할 때 발생하는 교통카드 데이터를 분석한 결과다.\
    구성역(경기 용인 기흥구)에서 탑승한 뒤 동탄역에서 내리는 구간의 통행시간이 가장 많이 줄었다. 58.0분에서 19.3분으로 단축되며 66.8% 감소 효과가 나타났다. 일평균 통행량이 가장 많은 수서~동탄 구간도 대폭 줄어들었다. 퇴근길에 해당하는 ‘수서역 승차, 동탄역 하차’ 구간은 99.3분 걸리던 것이 35.2분으로 개선됐다. 1시간가량 ‘여가 시간’을 확보하는 효과다.\
    GTX-A노선 정차역이 아닌 곳으로 이동할 때도 단축 효과가 있었다. 동탄역에서 대치역으로 이동할 경우 74.0분 걸리던 것이 46.8분으로 줄었다. 잠실역(78.8 → 54.0분), 한티역(73.7 → 50.1분), 삼성역(73.6 → 56.5분) 등 회사 밀집 지역으로 이동하는 데 걸리는 시간이 7~23분가량 단축됐다. 정차역 주변 지역을 광범위하게 연계하는 ‘광역교통수단’으로써의 역할을 톡톡히 하고 있는 것을 입증한 셈이다.\
    GTX 정차역에 대한 접근성은 개선해야 한다는 평가다. 수서역은 이용객의 79.0%가 GTX-A노선 외 대중교통을 이용한 것으로 나타났다. 구성역, 성남역, 수서역을 도시철도를 통해 이용하는 경우 26~29분이 걸렸다. 버스 또는 지하철을 2회 이상 환승하는 빈도도 8%대로 적지 않았다. 국토연구원 관계자는 “GTX-A노선을 이용할 수 있는 지역과 그렇지 않은 지역의 상대적 격차가 증가해 형평성 문제가 발생할 수 있다”고 분석했다.\
    ◆정차역 일대 집값 ‘들썩’…가까울수록 더 많이 올라 TX-A노선 도입 계획이 부동산 가격에 유의미한 영향을 미쳤다는 게 국토연구원의 분석이다. 연구원은 기본계획이 고시된 2017년을 기준으로 전후 3년간 아파트값을 비교했다. GTX 정차역 반경 1㎞ 내 지역과 그 외 지역을 대조했다.\
    가장 큰 영향을 받은 지역은 동탄역 일대다. 병점역 주변 단지와 비교해 집값 상승률이 29.2% 높았다. 구성역(비교군 신갈·기흥·상갈역)은 26.9%, 수서역(용산역) 11.9%, 성남역(수내·정자·미금·오리역) 3.8% 수준이다 \
    국토교통부 실거래가 공개시스템에 따르면 동탄역 반경 500m 내에 있는 ‘동탄역시범 더샵센트럴시티’ 전용면적 84㎡는 지난달 18일 14억4500만원(10층)에 거래됐다. 8년 전 10층 매물이 5억8000만원에 손바뀜한 것과 비교하면 9억원가량 오른 것이다. 같은 기간 반경 2㎞ 거리에 있는 ‘반도유보라 아이비파크3.0’ 전용 84㎡는 약 4억원(3억9500만 → 7억8000만원) 상승하는 데 그쳤다. 상승률도 따져봐도 50%P(149% vs 97%) 이상 차이가 난다.\
    GTX 개통 후 거래량도 꾸준히 늘고 있다. 동탄역은 청계·반송·영천·오산동 4개 행정동에 둘러싸여 있다. 지난달 이들 지역에서는 총 589건의 아파트 매매거래가 이뤄졌다. 1년 전 같은 기간에는 227건, 2년 전에는 119건에 불과했다. ‘10·15 주택시장 안정화 방안’ 발표 후 비규제 지역인 화성이 주목받고 있다는 점을 감안해도 거래량이 증가한 것은 분명하다. 지난 9월에는 354건의 거래가 체결됐다."),
]

# 텍스트를 조각으로 쪼개기 (옵션)
text_splitter = CharacterTextSplitter(chunk_size=200, chunk_overlap=20)
split_docs = text_splitter.split_documents(docs)

# 랭체인이 지원하는 Chromadb에 저장
db = Chroma.from_documents(split_docs, embedding_model)

# Retriever
retriever = db.as_retriever()

# PromptTemplate 작성
prompt_template = PromptTemplate(
    input_variables=['context', 'question'],
    template = """
      너는 친절하고 똑똑한 AI 어시스턴트야.
      아래 문서 내용을 참고해서 나의 질문에 정확하게 답을 해줘.
      문서 내용이 불충분한 경우 '문서에 해당 정보가 없어요' 라고 답변해.
      문맥:
      {context}
      질문:
      {question}
      답변은 5행 정도면 좋아
    """
)


# 체인 생성 (LCEL)

def format_docsFunc(docs):
  # 검색된 Document 리스트를 하나의 문자열로 합쳐 반환하는 헬퍼 함수
  return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {
        "context": retriever | format_docsFunc,
        "question": RunnablePassthrough()
    }
    | prompt_template
    | llm
    | StrOutputParser()
)

# 질문
query = "뉴욕증시는 무엇 때문에 하락 마감했어?"
result = rag_chain.invoke(query)
print("질문:",query)
print("답변:",result)

# 전체 구조 : 질문 -> 검색 -> 프롬프트 -> LLM -> 출력파싱

질문: 뉴욕증시는 무엇 때문에 하락 마감했어?
답변: 뉴욕증시는 장 초반 엔비디아 호실적과 9월 고용보고서에 힘입어 급등했지만, 약세로 반전하며 마감했습니다.

주요 하락 원인은 다음과 같습니다:
1.  AI 거품 우려가 지속되면서 기술주 투매가 일어났습니다.
2.  연방준비제도 관계자가 금융자산에 대해 급락 위험 경고를 한 것도 증시에 약세 압력을 주었습니다.
3.  2주 이상 일자리를 찾지 못한 '계속 실업수당' 청구자가 팬데믹 이후 4년 만에 최대 수준으로 늘어난 점도 전반적인 투자 심리에 부정적인 영향을 미쳤습니다.


In [19]:
# 멀티 실행 체인 : RunnableParallel을 사용해 여러 체인을 병렬로 처리 (과금 필요)
# 여건상 RunnableParallel 사용 불가. 그래서 여기서는 순차적으로 두 번 LLM 호출
llm_calm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)
llm_creative = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.9)
q = "AI가 뭔가요?"
out1 = llm_calm.invoke(q)
out2 = llm_creative.invoke(q)
print("1) 현실적인 답변 :", out1.content)
print("2) 창의적인 답변 :", out2.content)

1) 현실적인 답변 : AI는 **인공지능(Artificial Intelligence)**의 줄임말로, **컴퓨터나 기계가 인간처럼 생각하고, 학습하고, 문제를 해결하며, 의사결정을 내리는 능력을 갖추도록 만드는 기술**을 말합니다.

쉽게 말해, **기계가 사람처럼 똑똑해지는 것**이라고 생각하시면 됩니다.

**핵심적인 특징은 다음과 같습니다:**

1.  **학습 능력 (Learning):** 방대한 데이터를 분석하여 스스로 규칙이나 패턴을 찾아내고, 이를 통해 새로운 정보를 학습합니다. (예: 머신러닝, 딥러닝)
2.  **추론 능력 (Reasoning):** 학습한 지식과 정보를 바탕으로 논리적으로 생각하고 결론을 도출합니다.
3.  **문제 해결 능력 (Problem Solving):** 주어진 문제를 이해하고, 해결책을 찾아 실행합니다.
4.  **지각 능력 (Perception):** 이미지, 음성, 텍스트 등 다양한 형태의 정보를 인식하고 이해합니다.
5.  **언어 이해 능력 (Language Understanding):** 사람의 언어를 이해하고, 자연스럽게 대화하거나 번역할 수 있습니다.

**AI가 우리 생활에서 어떻게 활용되는지 몇 가지 예를 들어볼까요?**

*   **음성 비서:** 스마트폰의 시리(Siri), 구글 어시스턴트, 빅스비 등이 음성 명령을 이해하고 실행합니다.
*   **추천 시스템:** 넷플릭스, 유튜브, 온라인 쇼핑몰 등에서 사용자의 취향에 맞는 콘텐츠나 상품을 추천해 줍니다.
*   **자율주행차:** 주변 환경을 인식하고 스스로 판단하여 운전합니다.
*   **번역 앱:** 다른 언어를 실시간으로 번역해 줍니다.
*   **얼굴 인식/이미지 인식:** 스마트폰 잠금 해제, 사진 속 인물 태그, 보안 시스템 등에 사용됩니다.
*   **챗봇:** 고객 상담이나 정보 제공을 위해 사람처럼 대화합니다.
*   **의료 분야:** 질병 진단을 돕거나 신약 개발에 활용됩니다.

결론적으로, AI는 컴퓨터가 인간처럼 생각